# **Purpose**

**Task:** context + answer → question

This notebook fine-tunes `google/flan-t5-base` on a subset of SQuAD to generate a question given a context passage and an answer span. It's designed as a reusable template - swap `MODEL_NAME` and rerun to benchmark other Flan-T5 sizes.

**Runtime:** Runtime → Change runtime type → GPU (T4 is enough for `flan-t5-base` with the settings below).

## **Install dependencies**

In [ ]:
!pip install -qU \
 transformers==5.16.1 \
 evaluate==0.4.6 \
 rouge_score \
 sentencepiece==0.2.2 \
 accelerate==1.14.0 \
 sacrebleu

## **Imports**

In [7]:
import numpy as np
import torch
import transformers
import accelerate
import sentencepiece as spm
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from kaggle_secrets import UserSecretsClient
import os

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("SentencePiece:", spm.__version__)
print("Evaluate:", evaluate.__version__)


torch: 2.10.0+cu128
transformers: 5.16.1
accelerate: 1.14.0
SentencePiece: 0.2.2
Evaluate: 0.4.6


## **Load the API Keys and Tokens**


In [8]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Config**

Tweak these for your experiment. `TRAIN_SUBSET_SIZE` / `VAL_SUBSET_SIZE` control how much of SQuAD you use - start small to sanity-check the pipeline before scaling up.

In [20]:
MODEL_NAME = "google/flan-t5-base"      # try "google/flan-t5-small" if GPU memory/time is tight
MAX_INPUT_LENGTH = 384                  # context+answer prompt length
MAX_TARGET_LENGTH = 64                  # questions are short
TRAIN_SUBSET_SIZE = 500                # subset of SQuAD train split, set to None for full data
VAL_SUBSET_SIZE = 100
OUTPUT_DIR = "/content/flan-t5-base-qg"
SEED = 42

## **Load SQuAD and take a subset**

Uses the Hugging Face `squad` dataset. Swap to `"squad_v2"` if you also want unanswerable examples (note: `squad_v2` has empty answer lists for some examples, which the preprocessing below already handles gracefully).

In [9]:
raw = load_dataset("squad")

train_ds = raw["train"].shuffle(seed=SEED)
val_ds = raw["validation"].shuffle(seed=SEED)

if TRAIN_SUBSET_SIZE:
    train_ds = train_ds.select(range(TRAIN_SUBSET_SIZE))
if VAL_SUBSET_SIZE:
    val_ds = val_ds.select(range(VAL_SUBSET_SIZE))

print(train_ds)
print(val_ds)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 500
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 100
})


In [10]:
# Peek at one example
train_ds[0]

{'id': '573173d8497a881900248f0c',
 'title': 'Egypt',
 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.',
 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?',
 'answers': {'text': ['84%'], 'answer_start': [468]}}

## 5. Load tokenizer and model

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 6. Preprocessing

Builds the prompt: `"generate question: context: {context} answer: {answer}"` → target is the gold question.

**Tip:** for the "answer highlighting" trick used in a lot of QG literature, wrap the answer span inside the context with a marker (e.g. `<hl> {answer} <hl>`) before building the prompt — this can measurably improve which part of the context the model attends to.

In [12]:
def preprocess(examples):
    inputs = []
    for context, answers in zip(examples["context"], examples["answers"]):
        answer_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        prompt = f"generate question: context: {context} answer: {answer_text}"
        inputs.append(prompt)

    targets = examples["question"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

## 7. Metrics

ROUGE-L and BLEU as generation-quality proxies (standard in QG papers, though both have known limits for measuring semantic correctness — consider adding a round-trip QA consistency check later, as discussed in chat).

In [17]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # NEW: sanitize preds the same way labels are sanitized
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_rouge = [l.strip() for l in decoded_labels]
    decoded_labels_bleu = [[l.strip()] for l in decoded_labels]

    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels_rouge)
    bleu_result = bleu.compute(predictions=decoded_preds, references=decoded_labels_bleu)

    return {
        "rougeL": rouge_result["rougeL"],
        "bleu": bleu_result["score"],
    }

## 8. Training arguments

On Colab, set `fp16=True` if you have a T4/V100/A100 GPU (NVIDIA mixed precision). Adjust batch size / gradient accumulation if you hit out-of-memory errors.

In [18]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=4,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=torch.cuda.is_available(),   # mixed precision on Colab GPU
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",   # set to "wandb"/"tensorboard" if you use experiment tracking
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 9. Train

In [19]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rougel,Bleu
1,No log,1.627702,0.431186,17.996852
2,No log,1.572681,0.429593,17.079245
3,No log,1.639247,0.429907,16.663362
4,3.989977,1.661720,0.432067,16.559421


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=64, training_loss=3.825223386287689, metrics={'train_runtime': 175.659, 'train_samples_per_second': 11.386, 'train_steps_per_second': 0.364, 'total_flos': 850709366747136.0, 'train_loss': 3.825223386287689, 'epoch': 4.0})

## 10. Save the fine-tuned model

In [21]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/flan-t5-base-qg-demorun


In [ ]:
model.push_to_hub("your-username/flan-t5-base-qg")
model.push_to_hub("gauravdey2024/flan-t5-base-qg")
tokenizer.push_to_hub("your-username/flan-t5-base-qg")

Optional: mount Google Drive and copy the checkpoint there so it persists after the Colab runtime disconnects.

In [22]:
# Optional: save to Google Drive instead of losing it when the runtime disconnects
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {OUTPUT_DIR} /content/drive/MyDrive/flan-t5-qg-squad

NotImplementedError: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.

## 11. Sanity-check generations

In [23]:
sample = val_ds.select(range(5))
for ex in sample:
    answer_text = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    prompt = f"generate question: context: {ex['context']} answer: {answer_text}"
    input_ids = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
    ).input_ids.to(model.device)
    output_ids = model.generate(input_ids, max_length=MAX_TARGET_LENGTH, num_beams=4)
    generated_question = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"Answer:              {answer_text}")
    print(f"Gold question:       {ex['question']}")
    print(f"Generated question:  {generated_question}")
    print("-" * 80)

Answer:              1852
Gold question:       In what year did Massachusetts first require children to be educated in schools?
Generated question:  When was compulsory education established in Massachusetts?
--------------------------------------------------------------------------------
Answer:              1962
Gold question:       When were stromules discovered?
Generated question:  When were stromules first observed?
--------------------------------------------------------------------------------
Answer:              Horace Walpole
Gold question:       Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
Generated question:  Who was a major influence on the Gothic Revival?
--------------------------------------------------------------------------------
Answer:              several regional colleges and universities
Gold question:       In 1890, who did the university decide to team up with?
Generated question:  Where did the U

## Next steps

- Swap `MODEL_NAME` to `google/flan-t5-small` or `google/flan-t5-large` and rerun to compare sizes.
- Add the answer-highlighting (`<hl>` marker) preprocessing variant and compare against this baseline.
- Add a round-trip QA-consistency evaluation metric for a semantic-quality signal beyond ROUGE/BLEU.
- Reuse this notebook's data-loading/eval cells as a template for the decoder-only models (Qwen2.5, Llama-3.2, Phi-3.5) — those need causal-LM formatting and LoRA/QLoRA instead of the `Seq2SeqTrainer` setup here.